# Exploratory Data Analysis (EDA) & RFM Segmentation

**Project:** E-commerce Sales Data Analysis & Machine Learning  
**Author:** Muhammad Iqbal Fadel  
**Source:** `scripts/01_eda_run.py`

In [ ]:
%matplotlib inline

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10,5)

In [ ]:
ROOT = os.path.abspath('..')
DATA_PATH = os.path.join(ROOT, 'data', 'processed', 'Sales_Transaction_v4a_cleaned.csv')
OUTDIR = os.path.join(ROOT, 'outputs', 'figures', 'eda')
os.makedirs(OUTDIR, exist_ok=True)
CSV_OUTDIR = os.path.join(ROOT, 'outputs', 'data', 'analysis')
os.makedirs(CSV_OUTDIR, exist_ok=True)

print('Loading data from', DATA_PATH)
try:
    df = pd.read_csv(DATA_PATH, low_memory=False)
except Exception as e:
    print('Error loading CSV:', e)
    sys.exit(1)

# Ensure date and YearMonth
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
if 'YearMonth' not in df.columns and 'Date' in df.columns:
    df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)

# Ensure TotalAmount
if 'TotalAmount' not in df.columns and 'Price' in df.columns and 'Quantity' in df.columns:
    df['TotalAmount'] = df['Price'] * df['Quantity']

print('Data shape:', df.shape)

### 1) Monthly sales trend

In [ ]:
monthly = df.groupby('YearMonth')['TotalAmount'].sum().reset_index()
plt.figure(figsize=(12,5))
sns.lineplot(data=monthly, x='YearMonth', y='TotalAmount', marker='o')
plt.xticks(rotation=45)
plt.title('Monthly Sales Trend (TotalAmount)')
plt.tight_layout()
fn = os.path.join(OUTDIR, 'monthly_sales_trend.png')
plt.savefig(fn, dpi=150)
print('Saved', fn)
plt.clf()

### 2) Top 20 products by total sales

In [ ]:
if 'ProductName' in df.columns:
    prod = df.groupby('ProductName')['TotalAmount'].sum().nlargest(20).reset_index()
    plt.figure(figsize=(12,8))
    sns.barplot(data=prod, y='ProductName', x='TotalAmount', palette='viridis')
    plt.title('Top 20 Products by Total Sales')
    plt.xlabel('Total Sales (GBP)')
    plt.tight_layout()
    fn = os.path.join(OUTDIR, 'top20_products_by_sales.png')
    plt.savefig(fn, dpi=150)
    print('Saved', fn)
    plt.clf()

### 3) Top 15 countries by sales

In [ ]:
if 'Country' in df.columns:
    country = df.groupby('Country')['TotalAmount'].sum().nlargest(15).reset_index()
    plt.figure(figsize=(10,6))
    sns.barplot(data=country, x='TotalAmount', y='Country', palette='crest')
    plt.title('Top 15 Countries by Sales')
    plt.tight_layout()
    fn = os.path.join(OUTDIR, 'top15_countries_by_sales.png')
    plt.savefig(fn, dpi=150)
    print('Saved', fn)
    plt.clf()

### 4) Quantity distribution & TotalAmount boxplot

In [ ]:
if 'Quantity' in df.columns:
    plt.figure(figsize=(10,4))
    sns.histplot(df['Quantity'].clip(upper=df['Quantity'].quantile(0.99)), bins=50)
    plt.title('Quantity Distribution (clipped 99th percentile)')
    plt.tight_layout()
    fn = os.path.join(OUTDIR, 'quantity_distribution.png')
    plt.savefig(fn, dpi=150)
    print('Saved', fn)
    plt.clf()

if 'TotalAmount' in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=df['TotalAmount'].clip(lower=df['TotalAmount'].quantile(0.01), upper=df['TotalAmount'].quantile(0.99)))
    plt.title('Boxplot TotalAmount (clipped 1-99 percentile)')
    plt.tight_layout()
    fn = os.path.join(OUTDIR, 'totalamount_boxplot.png')
    plt.savefig(fn, dpi=150)
    print('Saved', fn)
    plt.clf()

### 5) RFM segmentation

In [ ]:
if {'CustomerNo', 'Date', 'TotalAmount'}.issubset(df.columns):
    rfm_base = df.dropna(subset=['CustomerNo', 'Date', 'TotalAmount']).copy()
    rfm_ref_date = rfm_base['Date'].max() + pd.Timedelta(days=1)
    freq_agg = 'nunique' if 'TransactionNo' in rfm_base.columns else 'count'
    rfm = rfm_base.groupby('CustomerNo').agg(
        Recency=('Date', lambda x: (rfm_ref_date - x.max()).days),
        Frequency=('TransactionNo', freq_agg),
        Monetary=('TotalAmount', 'sum')
    ).reset_index()

    rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'), 4, labels=[4, 3, 2, 1]).astype(int)
    rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
    rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)

In [ ]:
def segment_customer(row):
        if row['R_Score'] >= 3 and row['F_Score'] >= 3 and row['M_Score'] >= 3:
            return 'Champions'
        if row['R_Score'] >= 3 and row['F_Score'] >= 2:
            return 'Loyal'
        if row['R_Score'] <= 2 and row['F_Score'] >= 3:
            return 'At Risk'
        if row['R_Score'] <= 2 and row['F_Score'] <= 2:
            return 'Hibernating'
        return 'Potential'

    rfm['Segment'] = rfm.apply(segment_customer, axis=1)
    rfm_summary = rfm.groupby('Segment').agg(
        Customers=('CustomerNo', 'nunique'),
        Avg_Recency=('Recency', 'mean'),
        Avg_Frequency=('Frequency', 'mean'),
        Avg_Monetary=('Monetary', 'mean')
    ).reset_index().sort_values('Customers', ascending=False)

    rfm_csv = os.path.join(CSV_OUTDIR, 'rfm_summary.csv')
    rfm_summary.to_csv(rfm_csv, index=False)
    print('Saved', rfm_csv)

    plt.figure(figsize=(10,5))
    sns.barplot(data=rfm_summary, x='Segment', y='Customers', palette='mako')
    plt.title('RFM Segment Counts')
    plt.xticks(rotation=30)
    plt.tight_layout()
    fn = os.path.join(OUTDIR, 'rfm_segment_counts.png')
    plt.savefig(fn, dpi=150)
    print('Saved', fn)
    plt.clf()

    pivot = rfm_summary.set_index('Segment')[['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']]
    plt.figure(figsize=(8,4))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGnBu')
    plt.title('RFM Segment Averages')
    plt.tight_layout()
    fn = os.path.join(OUTDIR, 'rfm_segment_averages_heatmap.png')
    plt.savefig(fn, dpi=150)
    print('Saved', fn)
    plt.clf()

print('All plots saved to', OUTDIR)